# 04 — Tools & Function Calling
### Turning Python functions into things a model can call

LangChain Tools are how you expose real actions (DB lookups, API calls,
searches) to a model or to your own code in a standardized way. This
notebook covers the `@tool` decorator, direct tool invocation, and the
difference between *calling a tool yourself* and *letting a model decide to
call it* — which is exactly the design decision `tools.py` and `pipeline.py`
make in SupportPilot.

## 4.1 The `@tool` decorator

Decorating a function with `@tool` does two things: it wraps the function so
it satisfies the Runnable interface (so it can be `.invoke()`d directly), and
it auto-generates a JSON schema from the function's type hints and docstring
— that schema is what a model would see if you handed the tool to it.

In [ ]:
from langchain_core.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    'Add two integers together.'
    return a + b

print(add_numbers.name)
print(add_numbers.description)
print(add_numbers.args)          # the auto-generated schema
print(add_numbers.invoke({"a": 3, "b": 4}))


Note that the **docstring becomes part of the schema**. If you hand this
tool to a real model, the docstring is literally what the model reads to
decide whether and how to call it — this is why `tools.py` writes real,
specific docstrings rather than one-liners like "gets stuff":

```python
@tool
def get_customer_tool(customer_id: str) -> dict:
    'Look up a ShopStream India customer profile by customer_id (e.g. CUST001).'
    return database.get_customer(customer_id) or {}
```

## 4.2 The project's real tools

Let's import the actual tools from the project and inspect them the same
way. First, make sure the database is seeded (tools.py's KB retriever also
builds a FAISS index on import, so this cell may take a moment).

In [ ]:
import database
database.init_db(force=True)
print("DB seeded")


In [ ]:
from tools import get_customer_tool, get_orders_tool, retrieve_kb_tool

for t in [get_customer_tool, get_orders_tool, retrieve_kb_tool]:
    print(f"{t.name}: {t.description}")
    print(f"  args: {t.args}")
    print()


In [ ]:
# Calling a tool directly -- this is what pipeline.py does. No model involved.
customer = get_customer_tool.invoke({"customer_id": "CUST001"})
print(customer)

orders = get_orders_tool.invoke({"customer_id": "CUST001"})
print(f"\n{len(orders)} orders found, most recent: {orders[0] if orders else None}")


In [ ]:
kb_results = retrieve_kb_tool.invoke({"query": "refund for a damaged item"})
for r in kb_results:
    print(f"{r['source_doc']} — {r['section']}")


## 4.3 Direct invocation vs. letting a model choose

So far every tool call above was **you** deciding which tool to call, with
what arguments, in what order. That's "direct invocation" — treating a Tool
as just a well-documented function.

The other pattern is **binding tools to a model** and letting the model
decide which tool(s) to call based on the conversation. This is what agentic
systems (Week 4's AutoGen/CrewAI content) rely on.

In [ ]:
from langchain_core.messages import HumanMessage

# We can't demonstrate real tool-calling decisions without a tool-calling-capable
# model (FakeListChatModel doesn't support it), but here's the real pattern:
#
#   from langchain_anthropic import ChatAnthropic
#   llm = ChatAnthropic(model="claude-sonnet-5", max_tokens=1000)
#   llm_with_tools = llm.bind_tools([get_customer_tool, get_orders_tool, retrieve_kb_tool])
#   response = llm_with_tools.invoke([HumanMessage(content="What's the status of CUST001's orders?")])
#   print(response.tool_calls)
#   # -> [{'name': 'get_orders_tool', 'args': {'customer_id': 'CUST001'}, 'id': '...'}]
#
# The model decides *which* tool and *what arguments* -- you'd then execute
# the call yourself (or via an AgentExecutor) and feed the result back.
print("See the commented pattern above -- requires a real tool-calling model.")


## 4.4 Why SupportPilot invokes tools directly instead of via an agent

`pipeline.py` never calls `.bind_tools()` or lets a model decide which tool
to call next. Every tool call is direct:

```python
def _account_lookup(customer_id):
    customer = get_customer_tool.invoke({"customer_id": customer_id})
    ...
```

This is a deliberate choice, not a missed feature. The ticket workflow's
step order is fixed by the business process: classify, then look up account
+ retrieve knowledge, then draft, then validate, then escalate. There's
nothing for a model to legitimately decide about *which* tool runs next —
letting a model pick would only add a new way for the pipeline to behave
unpredictably, with no upside.

Contrast this with Week 4's SupportPilot-as-multi-agent-system framing (the
version built with AutoGen/CrewAI in the original course capstone), where an
agent *does* have discretion — e.g. deciding whether a ticket needs KB
retrieval at all vs. being pure account-status lookup. That's the
"Framework comparison guide" distinction from the course: LangChain here is
used as a pipeline/graph-based tool (fixed structure), not as an autonomous
decision-maker.

## Exercise

1. Write a new tool, `get_prior_ticket_count_tool`, wrapping
   `database.get_prior_ticket_count`. Give it a docstring specific enough
   that a model reading only the docstring would know what it does and what
   argument to pass.
2. Invoke it directly for `CUST001`.
3. Imagine binding all four tools (the three existing ones plus yours) to a
   real model and asking it "has CUST001 contacted us before, and what
   should I tell them about their orders?" — write out (in a markdown cell,
   no need to run it) what you'd expect `response.tool_calls` to contain.